# **Mouting Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Installing Model For Face Detection**

In [ ]:
!pip install facenet-pytorch tqdm opencv-python

# **Imports**

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from facenet_pytorch import MTCNN
import torch


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn = MTCNN(keep_all=True, device=device)

# **Video Preprocessing**

In [ ]:
def extract_and_crop_largest_face(video_path, output_dir, num_frames=4):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, frame_count - 1, num_frames, dtype=int)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    save_dir = os.path.join(output_dir, video_name)
    os.makedirs(save_dir, exist_ok=True)

    for i, idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        boxes, _ = mtcnn.detect(frame_rgb)


        if boxes is None or len(boxes) == 0:
            continue


        largest_box = max(boxes, key=lambda b: (b[2]-b[0]) * (b[3]-b[1]))
        x1, y1, x2, y2 = [int(b) for b in largest_box]
        face = frame[y1:y2, x1:x2]

        if face.size == 0:
            continue

        face_path = os.path.join(save_dir, f"face_{i:03d}.jpg")
        cv2.imwrite(face_path, face)

    cap.release()


# **Storing The Extracted Frames**

In [ ]:
input_root = "/content/drive/MyDrive/DeepfakeDataset/FaceForensics_subset"
output_root = "/content/drive/MyDrive/DeepfakeDataset/FaceForensics_Frames"

categories = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures", "original"]

for cat in categories:
    input_dir = os.path.join(input_root, cat)
    output_dir = os.path.join(output_root, cat)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Processing {cat}...")
    for video_file in tqdm(os.listdir(input_dir), desc=f"{cat}"):
        if video_file.endswith(".mp4"):
            video_path = os.path.join(input_dir, video_file)
            extract_and_crop_largest_face(video_path, output_dir)


Processing Deepfakes...


Deepfakes: 100%|██████████| 199/199 [05:13<00:00,  1.57s/it]


Processing Face2Face...


Face2Face: 100%|██████████| 200/200 [05:14<00:00,  1.57s/it]


Processing FaceShifter...


...

FaceSwap: 100%|██████████| 200/200 [06:08<00:00,  1.84s/it]


Processing NeuralTextures...


NeuralTextures: 100%|██████████| 200/200 [05:56<00:00,  1.78s/it]


Processing original...


original: 100%|██████████| 1000/1000 [29:42<00:00,  1.78s/it]
